# Hands-on: Train a deterministic AIFS with Anemoi

In this session we train a small AIFS-style model from scratch with [anemoi-training](https://anemoi.readthedocs.io/projects/training/en/latest/). Almost everything about a training is set in a **configuration** (a set of YAML files), so most of this notebook is about reading that configuration. Then we train a model on a small dataset and look at the results.

**Learning objectives.** By the end of this session you will be able to:
- explain how an anemoi-training config is built from config groups, and change any setting from the command line
- find the most important settings: data, dataloader, graph, model, training
- train a model and follow it in MLflow
- find your checkpoint and run a forecast with it

In the next session we turn this deterministic model into an ensemble (CRPS) model, `training_aifs-ens.ipynb`.

**Resources**
- [anemoi-training documentation](https://anemoi.readthedocs.io/projects/training/en/latest/)
- [anemoi-datasets documentation](https://anemoi.readthedocs.io/projects/datasets/en/latest/)
- [Lang et al. 2024, AIFS - ECMWF's data-driven forecasting system](https://arxiv.org/abs/2406.01465)

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf

from helpers import config_diff, load_config, run_training

# The toy dataset and where the trainings write their output. On the course server nothing needs to change here.
DATASET = "/datasets/aifs-ea-an-oper-0001-mars-o48-2010-2022-6h-v1-toy.zarr"
OUTPUT_ROOT = Path.home() / "anemoi-output"

# Settings passed to every training and config in this notebook (Hydra "overrides", see below)
OVERRIDES = [f"system.input.dataset={DATASET}", f"system.output.root={OUTPUT_ROOT}/"]

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, training will be very slow")

## 1. The toy dataset

We train on a small version of the ERA5 training data of the AIFS: the **O48** grid (10,944 points, about 200 km apart), every 6 hours from 2010 to 2022, and only 23 variables. Anemoi datasets are Zarr stores, opened with `anemoi.datasets.open_dataset`.

In [ ]:
from anemoi.datasets import open_dataset

ds = open_dataset(DATASET)
print(f"shape (dates, variables, ensemble members, grid points): {ds.shape}")
print(f"dates: {ds.dates[0]} to {ds.dates[-1]}, every {ds.frequency}")
print(f"variables: {ds.variables}")

In [ ]:
import cartopy.crs as ccrs

date_index = 0
fig, axes = plt.subplots(1, 2, figsize=(14, 4), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, variable in zip(axes, ["2t", "z_500"]):
    values = ds[date_index, ds.name_to_index[variable], 0]
    im = ax.scatter(ds.longitudes, ds.latitudes, c=values, s=2, cmap="RdYlBu_r", transform=ccrs.PlateCarree())
    ax.coastlines()
    ax.set_title(f"{variable} on {ds.dates[date_index]}")
    plt.colorbar(im, ax=ax, shrink=0.8)

The variables play different roles in the model:

| Role | Variables | Input? | Output? |
|---|---|---|---|
| **prognostic** | `2t`, `tcw`, `t_850`, `z_1000`, `z_700`, `z_500`, `z_300`, `z_250` | yes | yes |
| **diagnostic** | `tp`, `cp` (precipitation) | no | yes |
| **forcing** | land-sea mask `lsm`, orography `z`, `sdor`, `slor`, insolation and the sine/cosine of latitude, longitude, time of day and day of year | yes | no |

The forcings are known in advance (constants or functions of time), so the model does not need to predict them.

**Question:** why are precipitation variables diagnostic instead of prognostic?

## 2. How a training config is organised

anemoi-training uses [Hydra](https://hydra.cc) to build the config from several files. The `configs/` folder has one sub-folder per **config group**, and each group contains alternatives:

```
configs/
├── deterministic_o48.yaml   <- the config we train with (session 1)
├── crps_o48.yaml            <- the ensemble config (session 2)
├── data/         toy.yaml, zarr.yaml, ...
├── dataloader/   toy_o48.yaml, native_grid.yaml, ...
├── graph/        multi_scale.yaml, encoder_decoder_only.yaml, ...
├── model/        graphtransformer.yaml, gnn.yaml, transformer.yaml, ...
├── training/     default.yaml, ensemble.yaml, ...
├── diagnostics/  evaluation.yaml, ...
└── system/       course.yaml, ...
```

The top of our config, the **defaults list**, picks one file from each group. Everything below the defaults list overrides values from those files.

In [ ]:
print(Path("configs/deterministic_o48.yaml").read_text())

`load_config` builds the full config the same way `anemoi-training train` does. It also fills in references such as `${data.resolution}`. The result is a nested dictionary with one section per config group.

In [ ]:
config = load_config("deterministic_o48", OVERRIDES)
print(list(config.keys()))

Any setting can be changed without editing a file, with an **override** `section.key=value`. `OVERRIDES` above already uses this to set the dataset path. For example:

In [ ]:
small = load_config("deterministic_o48", OVERRIDES + ["model.num_channels=128", "training.max_steps=50"])
print(small.model.num_channels, small.training.max_steps)

On the command line, the same overrides go after the config name:

```bash
anemoi-training train --config-path=$PWD/configs --config-name=deterministic_o48 model.num_channels=128 training.max_steps=50
```

## 3. The most important settings

### 3.1 `data`: what goes in and out of the model
The `data` section lists the forcing and diagnostic variables (all others are prognostic) and how each variable is **normalised** before entering the model: `mean-std` subtracts the mean and divides by the standard deviation, `std` only divides, `max` divides by the maximum, `none` leaves the values unchanged. The statistics come from the dataset.

In [ ]:
print(OmegaConf.to_yaml(config.data))

### 3.2 `dataloader`: which samples, and how many at a time
The dataloader splits the dataset into periods: training (2010-2020), validation (2021) and test (2022). A training **sample** is a short sequence of states: `training.multistep_input` input states (here 2: *t*-6h and *t*) plus the target (*t*+6h).

In [ ]:
for key in ["batch_size", "limit_batches", "training", "validation"]:
    print(f"--- {key}\n{OmegaConf.to_yaml(config.dataloader[key])}")
print("multistep_input:", config.training.multistep_input)

`limit_batches.training` cuts an "epoch" short to 100 batches, so that the model is validated (on 8 batches), plotted and checkpointed every 100 steps.

**Question:** about how many training samples are there? With a batch size of 2 on one GPU, how many steps would it take to see each of them once? What fraction of the data does our training of `training.max_steps` steps see?

### 3.3 `graph`: where the model computes
The AIFS is an **encoder-processor-decoder** model:
- the **encoder** maps the data on the O48 grid onto a coarser **hidden** mesh
- the **processor** does most of the work on the hidden mesh (8 transformer layers here)
- the **decoder** maps the result back onto the O48 grid

The graph defines the nodes of the grid and the hidden mesh, and the edges along which information flows: data → hidden (encoder), hidden ↔ hidden (processor) and hidden → data (decoder). Here the hidden mesh is a refined icosahedron.

In [ ]:
print(OmegaConf.to_yaml(config.graph.nodes))

In [ ]:
# Build the graph like the training does, and look at it
from anemoi.graphs.create import GraphCreator

graph_config = config.graph.copy()
graph_config.nodes.data.node_builder.dataset = DATASET
graph = GraphCreator(config=graph_config).create()
print(graph)

In [ ]:
def lonlat(nodes):
    """Node coordinates (stored in radians) in degrees, longitudes in [-180, 180)."""
    lat, lon = np.rad2deg(graph[nodes].x.cpu().numpy()).T
    return (lon + 180) % 360 - 180, lat

fig, ax = plt.subplots(figsize=(14, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.scatter(*lonlat("data"), s=1, c="grey", label=f"data nodes ({graph['data'].num_nodes})")
ax.scatter(*lonlat("hidden"), s=6, c="tab:red", label=f"hidden nodes ({graph['hidden'].num_nodes})")
ax.coastlines()
ax.set_extent([-30, 50, 25, 75])
ax.legend(loc="lower left")

### 3.4 `model`: the neural network
The `model` section chooses the classes of the encoder, processor and decoder, and their size. `num_channels` is the length of the vector the model keeps for each node (the operational AIFS uses 1024), and `processor.num_layers` the number of processor layers (16 in the operational AIFS).

Two more details:
- `residual`: the model predicts the change from the last input state rather than the new state itself (a skip connection)
- `bounding`: e.g. `ReluBounding` on `tp` so that the predicted precipitation is never negative

In [ ]:
for key in ["num_channels", "model", "residual", "bounding"]:
    print(f"--- {key}\n{OmegaConf.to_yaml(config.model[key]) if OmegaConf.is_config(config.model[key]) else config.model[key]}")
for block in ["encoder", "processor", "decoder"]:
    print(f"{block}: {config.model[block]._target_}")
print("processor layers:", config.model.processor.num_layers)

### 3.5 `training`: loss, learning rate and length
The deterministic model is trained with a **mean squared error** loss (`MSELoss`), summed over all output variables and grid points. **Scalers** weight its terms:
- `general_variable`: a weight per variable, e.g. temperature and geopotential count 3 times, precipitation much less
- `pressure_level`: upper levels count less than lower levels
- `node_weights`: each grid point counts by its area

In [ ]:
print(OmegaConf.to_yaml(config.training.training_loss))
print(OmegaConf.to_yaml(config.training.scalers.datasets.data.general_variable.weights))
for key in ["model_task", "max_steps", "precision", "lr"]:
    print(key, config.training[key])

The learning rate rises linearly for `lr.warmup` steps and then follows a cosine down to `lr.min` at `lr.iterations` steps:

In [ ]:
lr = config.training.lr
steps = np.arange(lr.iterations)
cosine = lr.min + 0.5 * (lr.rate - lr.min) * (1 + np.cos(np.pi * (steps - lr.warmup) / (lr.iterations - lr.warmup)))
schedule = np.where(steps < lr.warmup, lr.rate * steps / lr.warmup, cosine)

plt.figure(figsize=(6, 3))
plt.plot(steps, schedule)
plt.xlabel("training step")
plt.ylabel("learning rate")

### 3.6 `system` and `diagnostics`: where things go and what is logged
`system` holds the paths (dataset, output folder) and the hardware: here **one GPU**. `diagnostics` sets the plots made during validation, how often checkpoints are saved, and the logging. We log to **MLflow** in offline mode, which writes files into `system.output.root/logs/mlflow`.

In [ ]:
print(OmegaConf.to_yaml(config.system))
print(OmegaConf.to_yaml(config.diagnostics.log.mlflow))

## 4. Train

Now we train the model for `training.max_steps` steps. This takes about **5-10 minutes** on the course GPU. The first minute or so goes into building the graph and setting up the model.

You can also run the training in a terminal (from this folder), which keeps the notebook free:

```bash
export ANEMOI_BASE_SEED=42
anemoi-training train --config-path=$PWD/configs --config-name=deterministic_o48
```

`ANEMOI_BASE_SEED` fixes the random numbers (weight initialisation, order of the samples), so that a training can be repeated.

In [ ]:
run_training("deterministic_o48", OVERRIDES)

## 5. Follow the training in MLflow

MLflow records the losses and metrics. You can open the MLflow web interface in a terminal:

```bash
mlflow ui --backend-store-uri=$HOME/anemoi-output/logs/mlflow --port=5111
```

Here we read the same records with the MLflow Python API.

In [ ]:
client = mlflow.tracking.MlflowClient(tracking_uri=str(OUTPUT_ROOT / "logs" / "mlflow"))


def latest_run(run_name):
    """The most recent MLflow run with this name (the run_name set in the config)."""
    experiment = client.get_experiment_by_name("training-course")
    runs = client.search_runs([experiment.experiment_id], f"tags.mlflow.runName = '{run_name}'",
                              order_by=["attributes.start_time DESC"], max_results=1)
    if not runs:
        raise ValueError(f"No run called {run_name!r} yet")
    return runs[0]


def metric_history(run, metric):
    history = client.get_metric_history(run.info.run_id, metric)
    return pd.Series([m.value for m in history], index=[m.step for m in history], name=metric).sort_index()


def plot_metrics(runs, metrics, ylabel="loss"):
    """Plot the history of some metrics of one or more runs, one panel per metric."""
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4), squeeze=False)
    for ax, metric in zip(axes[0], metrics):
        for run in runs:
            values = metric_history(run, metric)
            ax.plot(values.index, values.values, ".-" if len(values) < 30 else "-", label=run.info.run_name)
        ax.set(title=metric, xlabel="training step", ylabel=ylabel, yscale="log")
        ax.legend()
    plt.tight_layout()

In [ ]:
run = latest_run("deterministic-o48")
print(f"run {run.info.run_name} ({run.info.run_id}), status {run.info.status}")
print("metrics:", sorted(run.data.metrics))

In [ ]:
# the loss on the training batches and on the validation batches (2021), averaged over each "epoch" of 100 steps
plot_metrics([run], ["train_multi_dataset_loss_epoch", "val_multi_dataset_loss_epoch"])

**What to look for**
- The training loss should fall quickly during the warm-up and then more slowly. It is noisy because every step sees only 2 samples.
- The validation loss is computed on 2021, which the model never trains on. If it stops falling while the training loss keeps falling, the model is overfitting (unlikely after so few steps).

In [ ]:
# Validation metrics at the end of the training: the MSE of each variable, of groups of variables
# (sfc = surface, pl = pressure level) and of all of them. Unlike the loss, these are in physical units
# (e.g. K² for temperature), and without the variable weights.
pd.Series({k: v for k, v in run.data.metrics.items() if "_metric/" in k}, name="value").sort_index()

In [ ]:
# The plots the training made during validation (diagnostics.plot in the config)
plot_dir = OUTPUT_ROOT / "plots" / run.info.run_id
plot_files = sorted(plot_dir.rglob("*sample*"), key=lambda f: f.stat().st_mtime)
print(f"{len(plot_files)} plots in {plot_dir}")

from IPython.display import Image, display
display(Image(filename=plot_files[-1]))  # the last one

The sample plot shows, for each variable, the input, the target, the prediction and their differences, for one validation sample. After so few steps the prediction is much smoother than the target.

## 6. Your checkpoint

The training saves two kinds of checkpoints in `system.output.root/checkpoint/<run id>/`:
- `last.ckpt` etc.: the full training state (weights, optimiser, learning-rate schedule), to continue the training
- `inference-last.ckpt` etc.: only the model and the metadata it needs to run a forecast. This is the kind of file you used in the inference session.

In [ ]:
checkpoint_dir = OUTPUT_ROOT / "checkpoint" / run.info.run_id
for file in sorted(checkpoint_dir.glob("*.ckpt")):
    print(f"{file.name:60} {file.stat().st_size / 1e6:6.0f} MB")

In [ ]:
from anemoi.utils.checkpoints import load_metadata

metadata = load_metadata(str(checkpoint_dir / "inference-last.ckpt"))
print("trained with:", metadata["config"]["training"]["model_task"])
print("time step:", metadata["config"]["data"]["timestep"])
print("variables:", metadata["dataset"]["data"]["variables"])

## 7. Tasks

### Task 1: Change one setting and compare

Train a second model that differs in **one** setting. Pick one (or split them between your neighbours and compare):

| Experiment | Override(s) |
|---|---|
| a. Higher learning rate | `training.lr.rate=5e-3` |
| b. Only one input state | `training.multistep_input=1` |
| c. Smaller / larger model | `model.num_channels=128` or `512` |
| d. Train for longer | `training.max_steps=3000` |
| e. Care more about 2 m temperature | `training.scalers.datasets.data.general_variable.weights.2t=10` |

Give each run its own name, so it shows up separately in MLflow:

```python
run_training("deterministic_o48", OVERRIDES + ["training.lr.rate=5e-3", "diagnostics.log.mlflow.run_name=lr-5e-3"])
```

Then compare the runs with `plot_metrics([latest_run("deterministic-o48"), latest_run("lr-5e-3")], [...])`.

**What to look for**
- **Loss curves:** which run reaches the lower validation loss after the same number of steps? Is the difference larger than the noise of the curves?
- **Cost:** how long does a step take? (The time per step is in the training output and in MLflow.)
- **Per variable:** compare the validation metrics of each variable (`..._metric/data/2t/...`) of both runs. Experiment e should improve `2t`: at the cost of which variables?
- **(b)** With one input state the model cannot see how fast things are changing. Does that matter after so few steps?

In [ ]:
# Task 1: your training and comparison

### Task 2 (optional): Run a forecast with your model

Open `../3-run-AIFS/inference-aifs-ens-o48-cpu.ipynb` and, after the download cell, point `ckpt_file` to your inference checkpoint:

```python
ckpt_file = Path.home() / "anemoi-output/checkpoint/<run id>/inference-last.ckpt"
```

Your model uses the same variables and grid as the O48 model of that notebook, so the rest runs unchanged. How does your forecast compare with the one of the provided model, e.g. in the verification section?

In [ ]:
print(checkpoint_dir / "inference-last.ckpt")